In [ ]:
import torch
import pandas as pd
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig


In [ ]:
# === 1) GPU Check ===
assert torch.cuda.is_available(), "Please enable a GPU runtime."
print("CUDA device:", torch.cuda.get_device_name(0))
torch.backends.cuda.matmul.allow_tf32 = True

# === 2) Model Choice (Quantized LLaVA-Med) ===
MODEL_ID = "chaoyinshe/llava-med-v1.5-mistral-7b-hf"

bnb_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

In [ ]:
# === 3) Load Model and Processor ===
print("Loading quantized LLaVA model...")
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_4bit,
    device_map="auto",
    low_cpu_mem_usage=True,
    attn_implementation=None,
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print("Model loaded successfully.")

# === 4) Build Prompt Function (Text Only) ===
def build_text_prompt(processor, question: str):
    """Builds a text-only chat prompt compatible with LLaVA"""
    messages = [{"role": "user", "content": [{"type": "text", "text": question}]}]
    if hasattr(processor, "apply_chat_template"):
        return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    elif hasattr(processor, "tokenizer") and hasattr(processor.tokenizer, "apply_chat_template"):
        return processor.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        return f"[INST] {question} [/INST]"

In [ ]:
# === 5) Inference Function (Text Only) ===
def generate_text_response(question, model, processor, max_new_tokens=256):
    # Prepare input prompt
    prompt = build_text_prompt(processor, question)
    inputs = processor(text=prompt, return_tensors="pt").to(model.device)

    # Generate output
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
        )

    # Decode output text
    decode_fn = getattr(processor, "decode", None) or processor.tokenizer.decode
    response = decode_fn(output_ids[0], skip_special_tokens=True)

    # Clean up (remove chat formatting if present)
    if "[/INST]" in response:
        response = response.split("[/INST]", 1)[-1].strip()

    return response.strip()

In [ ]:

import re

def parse_bracket_ner(text: str) -> dict:
    """
    Extract entities and their values dynamically from bracketed text.
    Example: [Category] value [Next] value2 ...
    Returns a dictionary: {Category: value, ...}
    """
    pattern = r'\[([^\]]+)\]\s*([^[]*)'
    matches = re.findall(pattern, text)

    ner_dict = {}
    for category, value in matches:
        value = value.strip().rstrip(". ,;:")

        # Small cleanup rules
        if category.lower() == "subject":
            value = re.sub(r'^(?:a|an|the)\s+', '', value, flags=re.IGNORECASE)
        if category.lower() == "effect":
            value = value.lower()
            if not value.endswith("development") and not value.endswith("development."):
                value = value + " development."
            else:
                if not value.endswith("."):
                    value += "."

        ner_dict[category.strip()] = value

    return ner_dict


def format_text2(ner_dict: dict) -> str:
    """
    Format the extracted NER dictionary into Text 2 style output.
    """
    lines = [""]
    for key, val in ner_dict.items():
        lines.append(f"        {key}: {val}")
    return "\n".join(lines)

In [ ]:
# === 6) Example Usage ===
if __name__ == "__main__":
    csv_path = "./data/LLaVA-Med/LLaVA_Med_Input_NER.csv"
    
    # question = """You are a named entity recognition(NER) system. Extract and categorize the subject, treatment and effect from the given text. 
    # Generate the answer in this format. Format: "[Adverse event] associated with [Subject] a case of chronic renal failure in a patient [Treatment] AZ [Effect] acute hemorrhagic gastritis"
    # Text: " """

    question = """
    You are a named entity recognition(NER) system. Extract and categorize the subject, treatment and effect from the given text. 

    Example 1:
    Text: After therapy with parenteral amiodarone (2300 mg in 3 days) and other measures, signs of congestive heart failure disappeared; subsequently the patient developed jaundice, marked increase in serum transaminase levels and fall in prothrombin time, and histologic changes of severe centrilobular necrosis were observed in hepatic biopsy.
    Response: 
        Treatment: parenteral amiodarone (2300 mg in 3 days) and other measures 
        Effect: jaundice, marked increase in serum transaminase levels and fall in prothrombin time, and histologic changes of severe centrilobular necrosis were observed in hepatic biopsy

    Example 2:
    Text: We recommend the cautious use of alum irrigation in patients with renal impairment and monitoring of serum aluminum levels to prevent excessive accumulation and toxicity.
    Response: 
        Subject: patients
        Treatment: alum irrigation 
        Effect: excessive accumulation and toxicity.

    Now, extract for the next text:
    Text:  """

    no_of_entries_to_process = 20

    df = pd.read_csv(csv_path)

    print(f"\nPrompt without Context: {question}")

    for index, row in df.iterrows():
        if index < 2:
            continue
        
        quest_context = question + row['data__context'] #+ '"'

        print(f"---------------------------------------")
        print(f"\n🧠 Context: {row['data__context']}")
        answer = generate_text_response(quest_context, model, processor)
        print(f"💬 LLaVA Response: {answer}")
        print(f"🪙 Gold Reference: {row['data__answers']}")

        ner = parse_bracket_ner(row['data__answers'])
        text2 = format_text2(ner)
        print(f"🕶️ Formatted Gold Reference: {text2}")

        if index == no_of_entries_to_process -1:
            break
